In [7]:
!pip install -q ultralytics mediapipe opencv-python

In [6]:
import cv2
import mediapipe as mp  # 스켈레톤을 추출하는 라이브러리
import numpy as np
import csv  # csv 저장을 위해 라이브러리 추가
import os   # 파일 경로 관리를 위해 라이브러리 추가

# MediaPipe Pose 모델 초기화
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    min_detection_confidence = 0.7,  # 감지 최소 신뢰도
    min_tracking_confidence = 0.3 # 추적 최소 신뢰도
)

# MediaPipe 그리기 유틸리티 초기화
mp_drawing = mp.solutions.drawing_utils

# 동영상 파일 경로
video_path = "aespa_test.mp4" # 분석할 동영상 파일 경로 입력
cap = cv2.VideoCapture(video_path)

# 출력 파일 이름 설정
output_filename = os.path.splitext(os.path.basename(video_path))[0] + "_skeleton.csv"
# CSV 파일 헤더 준비 (33개 랜드마크 * 4개 좌표)
landmarks = ['class'] + [f'{j}_{i}' for i in mp_pose.PoseLandmark._member_names_ for j in ('x', 'y', 'z', 'v')]

# 동영상 파일이 정상적으로 열렸는지 확인
if not cap.isOpened():
    print(f"오류: '{video_path}' 동영상을 열 수 없습니다.")
    exit()

print("스켈레톤 추출을 시작합니다. 종료하려면 'q' 키를 누르세요.")

with open(output_filename, 'w', newline='') as f:
    csv_writer = csv.writer(f)
    csv_writer.writerow(landmarks)  # 헤더 작성

    print(f"'{output_filename}' 파일에 스켈레톤 데이터 저장을 시작합니다.")
    frame_count = 0
    while cap.isOpened():
        # 동영상에서 프레임 읽기
        success, image = cap.read()

        if not success:
            print("동영상 스트림의 끝에 도달했거나 오류가 발생했습니다.")
            break

        # 성능 향상을 위해 이미지를 읽기 전용으로 표시
        image.flags.writeable = False
        # BGR 이미지를 RGB로 변환
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # MediaPipe Pose를 사용하여 포즈 감지 수행
        results = pose.process(image_rgb)

        # 이미지를 다시 쓰기 가능으로 변경
        image.flags.writeable = True

        # 감지된 스켈레톤(포즈 랜드마크)을 원본 이미지에 그리기
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                image,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                landmark_drawing_spec = mp_drawing.DrawingSpec(color = (245, 117, 66), thickness=2, circle_radius=2),
                connection_drawing_spec=mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2)
            )

            # 랜드마크 데이터 추출 및 csv 행으로 변환
            try:
                # 'dance' 클래스로 분류 (필요에 따라 변경 가능)
                class_name = "dance"

                # 모든 랜드마크의 x, y, z, v 값을 순서대로 리스트에 담기
                pose_row = list(np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten())

                # 클래스 이름과 랜드마크 데이터를 합쳐서 한 행으로 만듦
                row = [class_name] + pose_row
                
                # csv 파일에 한 행 쓰기
                csv_writer.writerow(row)

            except Exception as e:
                print(f"프레임 {frame_count} 처리 중 오류 발생: {e}")
                pass # 오류 발생 시 해당 프레임 건너뜀

        # 결과 영상 출력
        cv2.imshow('MediaPipe Pose Skeleton', image)

        frame_count += 1
        # 'q' 키를 누르면 루프 종료
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

# 자원 해제
cap.release()
cv2.destroyAllWindows()
pose.close()

print("스켈레톤 추출이 완료되었습니다.")

스켈레톤 추출을 시작합니다. 종료하려면 'q' 키를 누르세요.
'aespa_test_skeleton.csv' 파일에 스켈레톤 데이터 저장을 시작합니다.
스켈레톤 추출이 완료되었습니다.


In [8]:
! pip install scipy

In [13]:
# mediapipe 라이브러리를 사용하여 기본 스켈레톤 추출
# YOLO 라이브러리를 사용하여, 다중 객체 탐지 (Top_down 방식)
# 칼만 필터(Kalman Filter)를 사용하여, 각 관절의 다음 위치를 예측하고 보정하여 성능 개선
# 상태 예측 및 보간: 추적을 잠시 놓친 스켈레톤의 위치를 예측하여, 시각적 끊김 없이 보이도록 처리
# 슬롯 기반 할당(Slot-Based Assignment)' 개념을 도입하여 성능 향상
# 헝가리안 알고리즘(scipy.linear_sum_assignment)을 사용:
# - 매 프레임마다 새로 탐지된 사람들과 기존 '댄서 슬롯'들의 위치를 비교하여, 전체적으로 가장 거리가 가까운 최적의 짝을 확인
# 3D 칼만 필터: 개별 댄서의 물리적 움직임을 3차원 공간에서 부드럽고 정확하게 예측

import cv2
import mediapipe as mp
from ultralytics import YOLO
import numpy as np
import random
import csv
from collections import Counter
from scipy.optimize import linear_sum_assignment

# ----------------------------------
# 1. 클래스 및 함수 정의
# ----------------------------------
class KalmanFilter3D:
    """A simple Kalman filter for 3D point tracking."""
    def __init__(self, dt=1, std_acc=1, x_std_meas=0.1, y_std_meas=0.1, z_std_meas=0.1):
        self.state = np.zeros((6, 1))   # 상태 변수를 6차원으로 확장: [x, y, z, vx, vy, vz]
        # 상태 전이 행렬을 6x6으로 확장
        self.F = np.array([[1,0,0,dt,0,0], [0,1,0,0,dt,0], [0,0,1,0,0,dt],
                           [0,0,0,1,0,0], [0,0,0,0,1,0], [0,0,0,0,0,1]])
        # 측정 행렬을 3x6으로 확장
        self.H = np.array([[1,0,0,0,0,0], [0,1,0,0,0,0], [0,0,1,0,0,0]])
        self.Q = np.eye(6)*std_acc**2
        self.R = np.diag([x_std_meas**2, y_std_meas**2, z_std_meas**2])
        self.P = np.eye(6)
    def predict(self):
        self.state = np.dot(self.F, self.state)
        self.P = np.dot(np.dot(self.F, self.P), self.F.T) + self.Q
        return self.state
    def update(self, z):
        # 입력 z는 이제 3D 벡터 [x, y, z]
        S = np.dot(self.H, np.dot(self.P, self.H.T)) + self.R
        K = np.dot(np.dot(self.P, self.H.T), np.linalg.inv(S))
        self.state += np.dot(K, (z - np.dot(self.H, self.state)))
        self.P -= np.dot(np.dot(K, self.H), self.P)
        return self.state

class DancerSlot:
    """영구적인 댄서 슬롯의 모든 데이터를 저장하는 클래스"""
    def __init__(self, slot_id, initial_bbox):
        self.id = slot_id
        self.kalman_filters = [KalmanFilter3D() for _ in range(33)]
        self.landmarks = np.zeros((33, 4))
        self.bbox = initial_bbox
        self.disappeared_frames = 0
        self.is_active = True
        self.color = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))

# ----------------------------------
# 2. 초기 설정
# ----------------------------------
TARGET_PERSON_COUNT = 4
yolo_model = YOLO('yolov8m.pt')
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5)
video_path = "aespa_test.mp4"
cap = cv2.VideoCapture(video_path)

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)); fps = int(cap.get(cv2.CAP_PROP_FPS))
output_video_path = video_path.replace('.mp4', '_output.mp4'); out = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (frame_width, frame_height))
csv_output_path = video_path.replace('.mp4', '_skeletons_data.csv'); csv_file = open(csv_output_path, 'w', newline='', encoding='utf-8'); csv_writer = csv.writer(csv_file)
csv_writer.writerow(['frame', 'slot_id', 'landmark_id', 'x', 'y', 'z', 'visibility'])

dancer_slots = {}
MAX_DISAPPEARED_FRAMES = 15; frame_counter = 0

print(f"3D 칼만 필터 기반 추적 시스템을 시작합니다. 목표 인원수: {TARGET_PERSON_COUNT}명")

# ----------------------------------
# 3. 메인 루프: 프레임별 처리
# ----------------------------------
while cap.isOpened():
    success, frame = cap.read()
    if not success: break

    # STEP 1: YOLO로 모든 사람 '탐지' (track 대신 predict 사용)
    results = yolo_model.predict(frame, classes=[0], conf=0.4, verbose=False)
    detections = results[0].boxes.xyxy.cpu().numpy()
    
    # 신뢰도 점수 기반으로 상위 N개만 선택 (선택적)
    confs = results[0].boxes.conf.cpu().numpy()
    if len(detections) > TARGET_PERSON_COUNT:
        top_indices = np.argsort(confs)[-TARGET_PERSON_COUNT:]
        detections = detections[top_indices]

    # 💡 [오류 수정] 코드 구조 변경: 초기화와 할당 로직을 분리
    # --- 슬롯 초기화 로직 ---
    if len(dancer_slots) < TARGET_PERSON_COUNT:
        for i, bbox in enumerate(detections):
            # 이미 존재하는 슬롯의 개수를 ID로 사용
            slot_id = len(dancer_slots)
            if slot_id < TARGET_PERSON_COUNT:
                print(f"프레임 {frame_counter}: 댄서 슬롯 {slot_id} 초기화.")
                dancer_slots[slot_id] = DancerSlot(slot_id, bbox)
            else:
                break
    
    # --- 슬롯 할당 로직 (매 프레임 실행) ---
    matched_slot_ids = set()
    if detections.shape[0] > 0 and len(dancer_slots) > 0:
        active_slots = {sid: slot for sid, slot in dancer_slots.items() if slot.is_active}
        slot_keys = list(active_slots.keys())
        
        if len(slot_keys) > 0:
            # 비용 행렬 계산
            cost_matrix = np.zeros((len(slot_keys), len(detections)))
            for i, slot_id in enumerate(slot_keys):
                slot_center = ((active_slots[slot_id].bbox[0] + active_slots[slot_id].bbox[2]) / 2, (active_slots[slot_id].bbox[1] + active_slots[slot_id].bbox[3]) / 2)
                for j, det_box in enumerate(detections):
                    det_center = ((det_box[0] + det_box[2]) / 2, (det_box[1] + det_box[3]) / 2)
                    cost_matrix[i, j] = np.linalg.norm(np.array(slot_center) - np.array(det_center))
            
            # 헝가리안 알고리즘으로 최적 할당
            row_ind, col_ind = linear_sum_assignment(cost_matrix)
            
            # 할당된 슬롯 업데이트
            for r, c in zip(row_ind, col_ind):
                slot_id = slot_keys[r]
                matched_slot_ids.add(slot_id)
                person_slot = dancer_slots[slot_id]
                person_slot.disappeared_frames = 0
                person_slot.is_active = True
                person_slot.bbox = detections[c]
                
                x1, y1, x2, y2 = [int(coord) for coord in person_slot.bbox]
                
                padding = 0.15; box_w, box_h = x2 - x1, y2 - y1
                x1_pad = max(0, int(x1 - box_w * padding)); y1_pad = max(0, int(y1 - box_h * padding))
                x2_pad = min(frame_width, int(x2 + box_w * padding)); y2_pad = min(frame_height, int(y2 + box_h * padding))
                person_crop = frame[y1_pad:y2_pad, x1_pad:x2_pad]

                if person_crop.shape[0] > 0 and person_crop.shape[1] > 0:
                    crop_rgb = cv2.cvtColor(person_crop, cv2.COLOR_BGR2RGB); pose_results = pose.process(crop_rgb)
                    if pose_results.pose_landmarks and len(pose_results.pose_landmarks.landmark) == 33:
                        crop_h, crop_w, _ = person_crop.shape
                        for i, landmark in enumerate(pose_results.pose_landmarks.landmark):
                            measured_x, measured_y = x1_pad + landmark.x * crop_w, y1_pad + landmark.y * crop_h
                            measured_z = landmark.z * crop_w

                            kf = person_slot.kalman_filters[i]
                            if np.all(kf.state[0:3] == 0):
                                kf.state[0], kf.state[1], kf.state[2] = measured_x, measured_y, measured_z
                            kf.predict()
                            updated_state = kf.update(np.array([[measured_x], [measured_y], [measured_z]]))
                            person_slot.landmarks[i] = [updated_state[0, 0], updated_state[1, 0], updated_state[2, 0], landmark.visibility]

    # 할당되지 않은 슬롯 (사라진 사람) 처리
    unmatched_slot_ids = set(dancer_slots.keys()) - matched_slot_ids
    for slot_id in unmatched_slot_ids:
        person_slot = dancer_slots[slot_id]
        person_slot.disappeared_frames += 1
        if person_slot.disappeared_frames > MAX_DISAPPEARED_FRAMES:
            person_slot.is_active = False
            continue
        
        for i in range(33):
            kf = person_slot.kalman_filters[i]; predicted_state = kf.predict()
            person_slot.landmarks[i, 0:3] = predicted_state[0:3, 0].T
        
        center_x = np.mean(person_slot.landmarks[:, 0]); center_y = np.mean(person_slot.landmarks[:, 1])
        if not (np.isnan(center_x) or np.isnan(center_y)): # 랜드마크가 모두 0일 경우 NaN 방지
            w, h = person_slot.bbox[2] - person_slot.bbox[0], person_slot.bbox[3] - person_slot.bbox[1]
            person_slot.bbox = [center_x - w/2, center_y - h/2, center_x + w/2, center_y + h/2]

    # 최종 시각화 및 데이터 저장
    for slot_id, person_slot in dancer_slots.items():
        if not person_slot.is_active: continue
        
        color = person_slot.color
        label = f"ID: {person_slot.id}"
        if person_slot.disappeared_frames > 0:
            color = tuple(c // 2 for c in color)
            label += " (Lost)"

        x1, y1, x2, y2 = [int(c) for c in person_slot.bbox]
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        landmarks_to_process = person_slot.landmarks
        for i, landmark in enumerate(landmarks_to_process):
            csv_writer.writerow([frame_counter, person_slot.id, i, landmark[0], landmark[1], landmark[2], landmark[3]])
        for landmark in landmarks_to_process:
            if landmark[3] > 0.5: cv2.circle(frame, (int(landmark[0]), int(landmark[1])), 3, color, -1)
        
        connections = mp_pose.POSE_CONNECTIONS
        for connection in connections:
            start_idx, end_idx = connection
            if landmarks_to_process[start_idx, 3] > 0.5 and landmarks_to_process[end_idx, 3] > 0.5:
                start_point = (int(landmarks_to_process[start_idx, 0]), int(landmarks_to_process[start_idx, 1])); end_point = (int(landmarks_to_process[end_idx, 0]), int(landmarks_to_process[end_idx, 1]))
                cv2.line(frame, start_point, end_point, color, 2)

    out.write(frame)
    cv2.imshow('Slot-Based High-Performance Tracking', frame)
    frame_counter += 1
    if cv2.waitKey(1) & 0xFF == ord('q'): break

# ----------------------------------
# 4. 종료 처리
# ----------------------------------
csv_file.close(); cap.release(); out.release(); cv2.destroyAllWindows(); pose.close()
print(f"처리 완료! 최종 결과가 '{output_video_path}'와 '{csv_output_path}'에 저장되었습니다.")


3D 칼만 필터 기반 추적 시스템을 시작합니다. 목표 인원수: 4명
프레임 1: 댄서 슬롯 0 초기화.
프레임 1: 댄서 슬롯 1 초기화.
프레임 1: 댄서 슬롯 2 초기화.
프레임 1: 댄서 슬롯 3 초기화.
처리 완료! 최종 결과가 'aespa_test_output.mp4'와 'aespa_test_skeletons_data.csv'에 저장되었습니다.


In [ ]:
!pip install pandas openpyxl

In [16]:
# 롱 포멧 방식의 결과를 사람의 가독성을 위해 와이드 포멧으로 변경하는 코드
import pandas as pd
import os
import mediapipe as mp

# --- 설정 ---
# 변환할 원본 데이터 파일 경로
input_csv_path = 'aespa_test_skeletons_data.csv' 

# 최종적으로 생성될 엑셀 보고서 파일 경로
output_excel_path = input_csv_path.replace('_skeletons_data.csv', '_skeletons_report.xlsx')

# MediaPipe의 PoseLandmark enum을 사용하여, ID를 실제 관절 이름으로 매핑합니다.
landmark_names = [name.name for name in mp.solutions.pose.PoseLandmark]

# --- 데이터 변환 로직 ---
def convert_long_to_wide_excel(csv_path, excel_path):
    """
    '롱 포맷' CSV 데이터를 읽어, slot_id별로 시트를 나눈 '와이드 포맷' 엑셀 파일로 변환합니다.
    이때, 열을 '관절 부위' 중심으로 정렬합니다. (예: x_NOSE, y_NOSE, z_NOSE, v_NOSE, ...)
    """
    if not os.path.exists(csv_path):
        print(f"오류: 원본 데이터 파일 '{csv_path}'를 찾을 수 없습니다.")
        return

    print(f"'{csv_path}' 파일을 읽는 중입니다...")
    df_long = pd.read_csv(csv_path)
    
    track_ids = df_long['slot_id'].unique()
    
    print(f"총 {len(track_ids)}개의 고유한 Track ID를 발견했습니다: {sorted(track_ids)}")
    print(f"'{excel_path}' 엑셀 파일 생성을 시작합니다...")

    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        for track_id in sorted(track_ids):
            print(f"  - Track ID: {track_id} 시트를 처리하는 중...")
            
            df_person = df_long[df_long['slot_id'] == track_id]
            
            df_wide = df_person.pivot_table(
                index='frame', 
                columns='landmark_id', 
                values=['x', 'y', 'z', 'visibility']
            )
            
            # 💡 [개선] 열 순서를 관절 중심으로 재정렬합니다.
            # 1. 원하는 순서대로 컬럼 리스트를 새로 생성합니다.
            ordered_columns_tuples = []
            for i in range(len(landmark_names)): # 0부터 32까지 관절 ID 순회
                for axis in ['x', 'y', 'z', 'visibility']: # 각 관절에 대해 x, y, z, v 순서로 추가
                    ordered_columns_tuples.append((axis, i))

            # 2. 생성된 순서대로 DataFrame의 열을 재정렬합니다.
            df_wide = df_wide[ordered_columns_tuples]

            # 3. 재정렬된 컬럼의 이름을 직관적으로 변경합니다.
            new_columns = []
            for axis, idx in df_wide.columns:
                landmark_name = landmark_names[idx]
                axis_name = 'v' if axis == 'visibility' else axis
                new_columns.append(f'{axis_name}_{landmark_name}')
            
            df_wide.columns = new_columns
            
            df_wide = df_wide.sort_index()

            sheet_name = f'slot_id_{track_id}'
            df_wide.to_excel(writer, sheet_name=sheet_name)

    print("\n변환 완료!")
    print(f"최종 보고서가 '{excel_path}' 경로에 성공적으로 저장되었습니다.")


# --- 스크립트 실행 ---
if __name__ == "__main__":
    convert_long_to_wide_excel(input_csv_path, output_excel_path)

'aespa_test_skeletons_data.csv' 파일을 읽는 중입니다...
총 4개의 고유한 Track ID를 발견했습니다: [0, 1, 2, 3]
'aespa_test_skeletons_report.xlsx' 엑셀 파일 생성을 시작합니다...
  - Track ID: 0 시트를 처리하는 중...
  - Track ID: 1 시트를 처리하는 중...
  - Track ID: 2 시트를 처리하는 중...
  - Track ID: 3 시트를 처리하는 중...

변환 완료!
최종 보고서가 'aespa_test_skeletons_report.xlsx' 경로에 성공적으로 저장되었습니다.
